In [9]:
from scipy import stats
from statsmodels.stats.weightstats import ztest
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import numpy as np
import statistics as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

data = pd.read_csv('nba_second_round_history.csv')
ts = pd.read_csv('nba_true_shooting_data.csv')
# hide 2026 season data since it's not complete and may skew the analysis
data = data[data['Season'] < 2026]
ts = ts[ts['Season'] < 2026]
# detect missing values
print(data.isnull().sum())
print(ts.isnull().sum())
data

Season                    0
Team                      0
Opponent                  0
Games played              0
Average Points scored     0
Average Points allowed    0
Offensive Rating          0
Defensive Rating          0
eFG%                      0
Opp eFG%                  0
TOV%                      0
Opp TOV%                  0
ORB%                      0
Opp ORB%                  0
FT/FGA                    0
Opp FT/FGA                0
Champion                  0
dtype: int64
Season      0
Team        0
Opponent    0
Team PTS    0
Team FGA    0
Team FTA    0
Team TS%    0
Opp PTS     0
Opp FGA     0
Opp FTA     0
Opp TS%     0
dtype: int64


,Season,Team,Opponent,Games played,Average Points scored,Average Points allowed,Offensive Rating,Defensive Rating,eFG%,Opp eFG%,TOV%,Opp TOV%,ORB%,Opp ORB%,FT/FGA,Opp FT/FGA,Champion
0,1984,BOS,NYK,7,111.0,103.0,113.9,105.7,0.498,0.484,14.2,16.0,38.7,35.0,0.285,0.296,True
1,1984,MIL,NJN,6,98.2,96.3,102.8,100.9,0.471,0.405,18.3,12.2,32.6,35.5,0.373,0.340,False
2,1984,LAL,DAL,5,120.6,106.2,121.9,107.4,0.570,0.454,12.8,11.9,37.8,35.1,0.200,0.232,False
3,1984,PHO,UTA,6,103.7,101.0,107.0,104.3,0.462,0.468,12.9,15.3,33.9,32.4,0.230,0.300,False
4,1985,BOS,DET,6,120.5,112.7,118.1,110.4,0.519,0.478,13.6,11.4,37.1,35.7,0.302,0.196,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,2024,MIN,DEN,7,102.7,97.6,113.8,108.1,0.539,0.517,11.5,12.4,22.8,21.1,0.201,0.173,False
164,2025,IND,CLE,5,117.6,114.2,116.7,113.3,0.585,0.490,13.3,11.7,18.0,28.8,0.220,0.295,False
165,2025,NYK,BOS,6,105.7,105.2,112.9,112.4,0.508,0.515,11.7,12.2,29.6,27.6,0.211,0.176,False
166,2025,OKC,DEN,7,115.4,106.3,114.6,105.6,0.518,0.481,9.7,15.1,24.9,28.0,0.208,0.249,True


In [10]:
# 1. Add Net Rating column (淨效率值 = 每100回合的得分 - 每100回合的失分) (better measure of team performance than point differential because it accounts for pace)
data['Net Rating'] = (data['Offensive Rating'] - data['Defensive Rating']).round(3)

# 2. Add Average Points Difference column
data['Average Points Difference'] = (data['Average Points scored'] - data['Average Points allowed']).round(3)

# 3. Add eFG% Difference column (eFG% 有效命中率 = 總命中數＋0.5*三分球的命中數）/ 總出手數)
data['eFG% Difference'] = ((data['eFG%'] - data['Opp eFG%']) * 100).round(3)

# 4. Add TOV% Difference column (TOV% 失誤率 = 失誤數 / 總出手數)
data['TOV% Difference'] = ((data['TOV%'] - data['Opp TOV%'])).round(3)

# 5. Add ORB% Difference column
data['ORB% Difference'] = (data['ORB%'] - data['Opp ORB%']).round(3)

# 6. Add FT/FGA Difference column
data['FT/FGA Difference'] = (data['FT/FGA'] - data['Opp FT/FGA']).round(3)

# 7. Add Actual Wins column
data['Actual Wins (%)'] = (4 / data['Games played'] * 100).round(3)

# 8. Add Pythagorean Wins column (a measure of dominance)
data['Pythagorean Wins (13.91) (%)'] = ((data['Average Points scored'] ** 13.91) / ((data['Average Points scored'] ** 13.91) + (data['Average Points allowed'] ** 13.91)) * 100).round(3)
data['Pythagorean Wins (16.5) (%)'] = ((data['Average Points scored'] ** 16.5) / ((data['Average Points scored'] ** 16.5) + (data['Average Points allowed'] ** 16.5)) * 100).round(3)

# 9. Add True Shooting Percentage Difference column (TS% 真實命中率 = 總得分 / (2 * (總出手數 + 0.44 * 罰球數)))
data['TS% Difference'] = ((ts['Team TS%'] - ts['Opp TS%'])).round(3)

data.drop(columns=['Average Points scored', 'Average Points allowed', 'Offensive Rating', 'Defensive Rating', 'eFG%', 'Opp eFG%', 'TOV%', 'Opp TOV%', 'ORB%', 'Opp ORB%', 'FT/FGA', 'Opp FT/FGA'], inplace=True)
# Save the modified dataset to a new CSV file
data.to_csv('nba_second_round_history_advance.csv', index=False)

In [11]:
data = pd.read_csv('nba_second_round_history.csv')
data = data[data['Season'] < 2026]

data['Pythagorean Wins (13.91) (%)'] = ((data['Average Points scored'] ** 13.91) / ((data['Average Points scored'] ** 13.91) + (data['Average Points allowed'] ** 13.91)) * 100).round(3)
data['Pythagorean Wins (16.5) (%)'] = ((data['Average Points scored'] ** 16.5) / ((data['Average Points scored'] ** 16.5) + (data['Average Points allowed'] ** 16.5)) * 100).round(3)

data['Team TS%'] = ts['Team TS%']
data['Opp TS%'] = ts['Opp TS%']

data['eFG%'] = (data['eFG%'] * 100).round(1)
data['Opp eFG%'] = (data['Opp eFG%'] * 100).round(1)

data['FT/FGA'] = (data['FT/FGA'] * 100).round(1)
data['Opp FT/FGA'] = (data['Opp FT/FGA'] * 100).round(1)

data.to_csv('nba_second_round_history_augmented.csv', index=False)
data

,Season,Team,Opponent,Games played,Average Points scored,Average Points allowed,Offensive Rating,Defensive Rating,eFG%,Opp eFG%,...,Opp TOV%,ORB%,Opp ORB%,FT/FGA,Opp FT/FGA,Champion,Pythagorean Wins (13.91) (%),Pythagorean Wins (16.5) (%),Team TS%,Opp TS%
0,1984,BOS,NYK,7,111.0,103.0,113.9,105.7,49.8,48.4,...,16.0,38.7,35.0,28.5,29.6,True,73.894,77.456,54.9350,53.9574
1,1984,MIL,NJN,6,98.2,96.3,102.8,100.9,47.1,40.5,...,12.2,32.6,35.5,37.3,34.0,False,56.753,57.990,54.0129,47.1452
2,1984,LAL,DAL,5,120.6,106.2,121.9,107.4,57.0,45.4,...,11.9,37.8,35.1,20.0,23.2,False,85.430,89.071,59.7550,50.4063
3,1984,PHO,UTA,6,103.7,101.0,107.0,104.3,46.2,46.8,...,15.3,33.9,32.4,23.0,30.0,False,59.073,60.714,51.2187,52.6627
4,1985,BOS,DET,6,120.5,112.7,118.1,110.4,51.9,47.8,...,11.4,37.1,35.7,30.2,19.6,False,71.725,75.104,57.8882,51.6914
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,2024,MIN,DEN,7,102.7,97.6,113.8,108.1,53.9,51.7,...,12.4,22.8,21.1,20.1,17.3,False,67.007,69.855,57.2051,55.1482
164,2025,IND,CLE,5,117.6,114.2,116.7,113.3,58.5,49.0,...,11.7,18.0,28.8,22.0,29.5,False,60.063,61.871,61.7907,54.7480
165,2025,NYK,BOS,6,105.7,105.2,112.9,112.4,50.8,51.5,...,12.2,29.6,27.6,21.1,17.6,False,51.648,51.955,54.4150,54.7933
166,2025,OKC,DEN,7,115.4,106.3,114.6,105.6,51.8,48.1,...,15.1,24.9,28.0,20.8,24.9,True,75.815,79.499,55.5708,53.2585
